# 哪些联赛的补时进球占比最高？

用 **Football Charts 开放数据集**（版本 2026.10，CC BY 4.0）复现
[football-charts.com/zh-cn/dataset](https://www.football-charts.com/zh-cn/dataset) 上的补时进球排名。

- 数据：92 个联赛最近三个已完赛季的 76,991 场比赛，含半场比分和每个进球的分钟
- DOI：[10.5281/zenodo.22295583](https://doi.org/10.5281/zenodo.22295583)（本版本：10.5281/zenodo.23348754）
- 本笔记本只用 pandas 和 matplotlib；从下载到出图约一分钟

In [ ]:
import hashlib, io, urllib.request, zipfile
from collections import defaultdict
from pathlib import Path

import pandas as pd

VERSION = "2026.10"
SHA256 = "08d1a0863d703550fe6e7fb1ef9faed4ea3cea95c664012373b1a1e1f159be10"   # football_charts_matches.csv
# 两个地址内容完全相同；zenodo.org 在中国大陆部分网络下可能打不开，所以先用镜像
MIRROR = "https://www.football-charts.com/zh-cn/files/football_charts_matches_v2026.10.zip"
ZENODO = "https://zenodo.org/records/23348754/files/football_charts_matches.csv?download=1"
CSV = Path("football_charts_matches.csv")

## 1. 下载并校验数据

文件只下载一次；SHA-256 一致，说明你拿到的数据和本页、Zenodo 上的版本逐字节相同。

In [ ]:
if not CSV.exists():
    try:
        raw = urllib.request.urlopen(MIRROR, timeout=120).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            CSV.write_bytes(z.read("football_charts_matches.csv"))
    except Exception as e:
        print("镜像下载失败，改用 Zenodo：", e)
        CSV.write_bytes(urllib.request.urlopen(ZENODO, timeout=300).read())

digest = hashlib.sha256(CSV.read_bytes()).hexdigest()
assert digest == SHA256, f"校验失败：{digest}"
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
print(f"SHA-256 校验通过；{len(df):,} 场比赛，{df.league.nunique()} 个联赛")
df.head(3)

## 2. 统计口径

`all_goal_times` 是一场比赛所有进球的分钟，如 `"12,45,78"`。数据源记录补时进球的方式决定了口径：

- **上半场补时**：45+2 分钟的进球被记成 **47**，不是 45+2。所以 46–48 分钟记为上半场补时（`45+`）。
  如果简单地在第 45 分钟切开，这些球都会被算进下半场，上半场进球占比因此被低估。
- **下半场补时**：91 分钟及以后记为 `90+`。本页排名只用这一项：它没有歧义。
- `111` 是“无进球”的占位值，`Not found` 等无法解析的值、超过 120 的分钟都不计入。
- 每个联赛只取最近一个完整赛季：跨年赛制取 `2025-2026`，自然年赛制取 `2025`。
- 进球数少于 200 的联赛不排名；进球分钟覆盖率（有分钟的进球 ÷ 比分里的进球）须在 95%–105% 之间。

In [ ]:
def minute_bin(raw):
    t = str(raw or "").strip()
    if not t or t == "111":
        return None
    if "+" in t:                      # 数据里也可能出现 "45+2" / "90+3" 这种写法
        try:
            base = int(t.split("+")[0])
        except ValueError:
            return None
        if base >= 90:
            return "90+"
        if base == 45:
            return "45+"
        t = str(base)
    try:
        m = int(t)
    except ValueError:
        return None
    if m > 120:
        return None
    for hi, b in ((15, "0-15"), (30, "15-30"), (45, "30-45"), (48, "45+"),
                  (60, "46-60"), (75, "60-75"), (90, "75-90")):
        if m <= hi:
            return b
    return "90+"

SEASONS = {"2025-2026", "2025"}
rows = []
for league, g in df[df.season.isin(SEASONS)].groupby("league"):
    bins = defaultdict(int)
    for raw in g.all_goal_times:
        for t in raw.split(","):
            b = minute_bin(t)
            if b:
                bins[b] += 1
    n = sum(bins.values())
    score_goals = pd.to_numeric(g.total_goals).sum()
    rows.append({
        "league": league, "league_name": g.league_name.iat[0], "country": g.country.iat[0],
        "season": g.season.iat[0], "matches": len(g), "goals": n,
        "goals_90plus": bins["90+"], "pct_90plus": round(100 * bins["90+"] / n, 1) if n else None,
        "goals_45plus": bins["45+"], "pct_45plus": round(100 * bins["45+"] / n, 1) if n else None,
        "minute_coverage_pct": round(100 * n / score_goals, 1) if score_goals else 0.0,
    })

table = pd.DataFrame(rows)
table = table[(table.goals >= 200) & table.minute_coverage_pct.between(95, 105)]
table = table.sort_values(["pct_90plus", "league"], ascending=[False, True]).reset_index(drop=True)
table.insert(0, "rank", table.index + 1)
total = table.goals.sum()
print(f"{len(table)} 个联赛，{total:,} 个进球；90 分钟后的进球占 {100 * table.goals_90plus.sum() / total:.1f}%")
table.head(12)

## 3. 加上中文联赛名，并与网页上的数字核对

中文名取自网页提供的汇总表；下载不到时用英文名，计算结果不受影响。

In [ ]:
AGG = "stoppage_time_by_league_v2026.10.csv"
try:
    src = AGG if Path(AGG).exists() else "https://www.football-charts.com/zh-cn/files/" + AGG
    published = pd.read_csv(src, encoding="utf-8-sig")
    table = table.merge(published[["league", "name_zh"]], on="league", how="left")
    same = (table.set_index("league").pct_90plus
            == published.set_index("league").pct_90plus.reindex(table.league)).all()
    print("与网页数字一致" if same else "与网页数字不一致，请检查数据版本")
except Exception as e:
    print("没有拿到汇总表，用英文名：", e)
    table["name_zh"] = table.league_name + "（" + table.country + "）"
table[["rank", "name_zh", "league_name", "goals", "pct_90plus"]].head(12)

## 4. 画图：最高的 20 个联赛和最低的 10 个联赛

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

# matplotlib 默认字体不含中文：使用系统里第一个可用的中文字体
installed = {f.name for f in font_manager.fontManager.ttflist}
for name in ("PingFang SC", "Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
             "Source Han Sans SC", "WenQuanYi Zen Hei", "Heiti TC", "Arial Unicode MS"):
    if name in installed:
        plt.rcParams["font.sans-serif"] = [name]
        break
plt.rcParams["axes.unicode_minus"] = False

shown = pd.concat([table.head(20), table.tail(10)])
colors = ["#dc2626" if lg == "china" else "#f59e0b" for lg in shown.league]
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh([f"{r} {n}" for r, n in zip(shown["rank"], shown.name_zh)], shown.pct_90plus, color=colors)
ax.invert_yaxis()
ax.set_xlabel("90 分钟后的进球占全部进球的比例（%）")
ax.set_title("补时进球占比：最高的 20 个和最低的 10 个联赛（2025/26 赛季）")
for y, v in enumerate(shown.pct_90plus):
    ax.text(v + 0.1, y, f"{v:.1f}%", va="center", fontsize=8)
fig.text(0.99, 0.005, "数据：Football Charts 开放数据集 v2026.10 · CC BY 4.0", ha="right", fontsize=8, color="#64748b")
fig.tight_layout()
fig.savefig("stoppage_time_by_league.png", dpi=150)
plt.show()

## 5. 保存结果

In [ ]:
table.to_csv("my_stoppage_time_by_league.csv", index=False, encoding="utf-8-sig")  # utf-8-sig：Excel 可直接打开
print("已保存 my_stoppage_time_by_league.csv")

## 引用

使用这份数据请注明来源。参考格式（GB/T 7714）：

> Football Charts. Football Match Results and Goal Timing, 92 Leagues (2023–2026) [DS/OL]. Version 2026.10. Zenodo, 2026. DOI: 10.5281/zenodo.23348754.

BibTeX：

```bibtex
@dataset{football_charts_2026_10,
  author    = {{Football Charts}},
  title     = {Football Match Results and Goal Timing, 92 Leagues (2023--2026)},
  version   = {2026.10},
  publisher = {Zenodo},
  year      = {2026},
  doi       = {10.5281/zenodo.23348754},
  url       = {https://doi.org/10.5281/zenodo.23348754}
}
```

许可：CC BY 4.0。需要更早的赛季（2020 年起）或通过 API 调用，见
[football-charts.com/zh-cn/developers](https://www.football-charts.com/zh-cn/developers)。